# Notebook 2: Advanced Excel to Pandas Techniques

In this notebook, we'll cover more advanced Excel functions that are commonly used in business:

- VLOOKUP → merge/join
- Pivot Tables → groupby
- Conditional formatting
- Multiple aggregations

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Load the data
df = pd.read_csv('../data/superstore_sales.csv')

# Add profit columns as we did in notebook 1
df['Profit'] = df['Sales'] - df['Cost']
df['Profit_Margin'] = (df['Profit'] / df['Sales']) * 100

print("✅ Data loaded successfully!")
print(f"Shape: {df.shape}")

## 1. VLOOKUP → Merge/Join

### Excel VLOOKUP:
```excel
=VLOOKUP(lookup_value, table_array, col_index_num, [range_lookup])
```

### Business Scenario:
We have product information in a separate table and want to add it to our sales data.

In [ ]:
# Create a product information table (like a separate Excel sheet)
product_info = pd.DataFrame({
    'Product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor'],
    'Category': ['Electronics', 'Accessories', 'Accessories', 'Electronics'],
    'Supplier': ['TechCorp', 'PeripheralPlus', 'PeripheralPlus', 'ScreenPro'],
    'Warranty_Months': [24, 6, 12, 36]
})

print("Product Information Table:")
product_info

### Excel VLOOKUP Example:
```excel
=VLOOKUP(B2, ProductInfo!$A$2:$D$5, 2, FALSE)
```
This looks up the Product in B2, finds it in the ProductInfo table, and returns the Category (column 2).

### Pandas Merge (Better than VLOOKUP!):

In [ ]:
# Merge sales data with product info
# This is like doing multiple VLOOKUPs at once!
df_enriched = df.merge(product_info, on='Product', how='left')

print("Data after merge:")
df_enriched.head(10)

### Why Merge is Better than VLOOKUP:

1. **All columns at once**: No need to write multiple VLOOKUP formulas
2. **No column counting**: Don't need to count which column number
3. **Can look left**: VLOOKUP only looks to the right
4. **More flexible**: Can merge on multiple columns
5. **Faster**: Especially with large datasets

In [ ]:
# Analyze sales by category
category_analysis = df_enriched.groupby('Category').agg({
    'Sales': ['sum', 'mean', 'count'],
    'Profit': 'sum'
}).round(2)

print("Sales Analysis by Category:")
category_analysis

## 2. Pivot Tables → GroupBy

### Excel Way:
- Select data → Insert → PivotTable
- Drag fields to Rows, Columns, Values

### Pandas Way:

### Example 1: Simple Pivot - Total Sales by Region

In [ ]:
# Excel: PivotTable with Region in Rows, Sum of Sales in Values
# Pandas:
sales_by_region = df_enriched.groupby('Region')['Sales'].sum().sort_values(ascending=False)

print("Total Sales by Region:")
print(sales_by_region)
print(f"\nTotal: ${sales_by_region.sum():,.2f}")

### Example 2: Two-Dimensional Pivot - Sales by Product and Region

In [ ]:
# Excel: PivotTable with Region in Rows, Product in Columns, Sum of Sales in Values
# Pandas:
pivot_product_region = pd.pivot_table(
    df_enriched,
    values='Sales',
    index='Region',
    columns='Product',
    aggfunc='sum',
    fill_value=0
).round(2)

print("Sales by Product and Region:")
print(pivot_product_region)
print("\nRow totals:")
print(pivot_product_region.sum(axis=1))

### Example 3: Multiple Aggregations (Advanced Pivot)

In [ ]:
# Multiple calculations at once
# Excel: Would need multiple PivotTables or calculated fields
# Pandas: Easy with groupby + agg

comprehensive_analysis = df_enriched.groupby('Product').agg({
    'Sales': ['sum', 'mean', 'min', 'max', 'count'],
    'Profit': ['sum', 'mean'],
    'Profit_Margin': 'mean'
}).round(2)

print("Comprehensive Product Analysis:")
comprehensive_analysis

### Example 4: Pivot with Multiple Grouping

In [ ]:
# Group by Region AND Category
region_category = df_enriched.groupby(['Region', 'Category']).agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Profit_Margin': 'mean'
}).round(2)

print("Sales & Profit by Region and Category:")
region_category

## 3. Advanced Filtering (IF Statements)

### Excel Way:
```excel
=IF(D2>1000, "High", IF(D2>100, "Medium", "Low"))
```

### Pandas Way:

In [ ]:
# Categorize sales into tiers
def categorize_sale(sales):
    if sales > 1000:
        return 'High'
    elif sales > 100:
        return 'Medium'
    else:
        return 'Low'

df_enriched['Sales_Tier'] = df_enriched['Sales'].apply(categorize_sale)

# Or use pandas cut() for better performance
df_enriched['Sales_Tier_v2'] = pd.cut(
    df_enriched['Sales'],
    bins=[0, 100, 1000, float('inf')],
    labels=['Low', 'Medium', 'High']
)

print("Sales distribution by tier:")
print(df_enriched['Sales_Tier'].value_counts().sort_index())

## 4. SUMIF and COUNTIF

### Excel Way:
```excel
=SUMIF(B:B, "Laptop", D:D)
=COUNTIF(C:C, "North")
```

### Pandas Way:

In [ ]:
# SUMIF: Sum sales where Product = Laptop
laptop_total = df_enriched[df_enriched['Product'] == 'Laptop']['Sales'].sum()
print(f"Total Laptop Sales (SUMIF): ${laptop_total:,.2f}")

# COUNTIF: Count records where Region = North
north_count = df_enriched[df_enriched['Region'] == 'North'].shape[0]
print(f"Number of North region sales (COUNTIF): {north_count}")

# AVERAGEIF: Average sales where Category = Electronics
electronics_avg = df_enriched[df_enriched['Category'] == 'Electronics']['Sales'].mean()
print(f"Average Electronics Sales (AVERAGEIF): ${electronics_avg:,.2f}")

## 5. Visualizing Pivot Data

### Heatmap: Sales by Product and Region

In [ ]:
# Create a heatmap of the pivot table
plt.figure(figsize=(10, 6))
sns.heatmap(pivot_product_region, annot=True, fmt='.0f', cmap='YlGnBu', cbar_kws={'label': 'Sales ($)'})
plt.title('Sales Heatmap: Product vs Region', fontsize=14, fontweight='bold')
plt.xlabel('Product', fontsize=12)
plt.ylabel('Region', fontsize=12)
plt.tight_layout()
plt.show()

### Stacked Bar Chart: Sales by Region and Category

In [ ]:
# Prepare data for stacked bar chart
pivot_region_category = pd.pivot_table(
    df_enriched,
    values='Sales',
    index='Region',
    columns='Category',
    aggfunc='sum'
)

# Create stacked bar chart
pivot_region_category.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='Set2')
plt.title('Sales by Region and Category (Stacked)', fontsize=14, fontweight='bold')
plt.xlabel('Region', fontsize=12)
plt.ylabel('Sales ($)', fontsize=12)
plt.legend(title='Category', fontsize=10)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Date Operations

### Excel Way:
- Use functions like MONTH(), YEAR(), WEEKDAY()

### Pandas Way:

In [ ]:
# Convert Date column to datetime
df_enriched['Date'] = pd.to_datetime(df_enriched['Date'])

# Extract date components (like Excel's MONTH() and YEAR())
df_enriched['Year'] = df_enriched['Date'].dt.year
df_enriched['Month'] = df_enriched['Date'].dt.month
df_enriched['Month_Name'] = df_enriched['Date'].dt.month_name()
df_enriched['Quarter'] = df_enriched['Date'].dt.quarter
df_enriched['Day_of_Week'] = df_enriched['Date'].dt.day_name()

print("Date components added:")
df_enriched[['Date', 'Year', 'Month', 'Month_Name', 'Quarter', 'Day_of_Week']].head()

In [ ]:
# Sales by month
monthly_sales = df_enriched.groupby('Month_Name')['Sales'].sum().reindex([
    'January', 'February', 'March', 'April', 'May', 'June'
])

plt.figure(figsize=(12, 6))
monthly_sales.plot(kind='line', marker='o', linewidth=2, markersize=8, color='darkblue')
plt.title('Monthly Sales Trend', fontsize=14, fontweight='bold')
plt.xlabel('Month', fontsize=12)
plt.ylabel('Sales ($)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Practice Exercises

Try these on your own:

1. Create a pivot table showing average profit by supplier and region
2. Find which product-region combination has the highest profit margin
3. Calculate what percentage of sales each category represents
4. Create a chart showing sales trend by quarter

### Solutions:

In [ ]:
# Exercise 1: Average profit by supplier and region
supplier_region_profit = pd.pivot_table(
    df_enriched,
    values='Profit',
    index='Supplier',
    columns='Region',
    aggfunc='mean'
).round(2)
print("Average Profit by Supplier and Region:")
print(supplier_region_profit)

# Exercise 2: Highest profit margin combination
product_region_margin = df_enriched.groupby(['Product', 'Region'])['Profit_Margin'].mean()
highest_margin = product_region_margin.idxmax()
print(f"\nHighest profit margin: {highest_margin[0]} in {highest_margin[1]} region")
print(f"Margin: {product_region_margin.max():.2f}%")

# Exercise 3: Category percentage of sales
category_sales = df_enriched.groupby('Category')['Sales'].sum()
category_pct = (category_sales / category_sales.sum() * 100).round(2)
print("\nCategory Sales Percentage:")
print(category_pct)

# Exercise 4: Quarterly sales trend
quarterly_sales = df_enriched.groupby('Quarter')['Sales'].sum()
plt.figure(figsize=(10, 6))
quarterly_sales.plot(kind='bar', color='coral')
plt.title('Quarterly Sales', fontsize=14, fontweight='bold')
plt.xlabel('Quarter', fontsize=12)
plt.ylabel('Sales ($)', fontsize=12)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Summary

In this notebook, you learned advanced Excel to Pandas conversions:

| Excel Function | Pandas Equivalent | Advantage |
|----------------|-------------------|------------|
| VLOOKUP | `.merge()` | All columns at once, more flexible |
| Pivot Table | `.groupby()` + `.pivot_table()` | More powerful aggregations |
| SUMIF | `df[condition]['col'].sum()` | Clearer logic |
| COUNTIF | `df[condition].shape[0]` | More intuitive |
| IF statements | `.apply()` or `pd.cut()` | Faster on large data |
| MONTH/YEAR | `.dt.month` / `.dt.year` | More date functions |

**Next:** In Notebook 3, we'll put it all together in a real business project!